In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from models import get_model , get_embeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


In [9]:
loader = TextLoader("my_data.txt")

doc = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size= 500,
    chunk_overlap= 50,
    
)

text_spliter = splitter.split_documents(doc)

In [40]:
embd_model = get_embeddings()
llm = get_model()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1779.58it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# vectore search by using ChromaDB

vectorstore=Chroma.from_documents(text_spliter,embd_model)
vectorstore_retreiver = vectorstore.as_retriever(search_kwargs={"k": 3})

In [12]:
# Keyword
keyword_retriever = BM25Retriever.from_documents(text_spliter)
keyword_retriever.k =  3

In [27]:
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from typing import List

class EnsembleRetriever(BaseRetriever):
    retrievers: list
    weights: list = None

    def _get_relevant_documents(self, query: str) -> List[Document]:
        weights = self.weights or [1/len(self.retrievers)] * len(self.retrievers)
        
        seen = {}
        for retriever, weight in zip(self.retrievers, weights):
            docs = retriever.invoke(query)
            for rank, doc in enumerate(docs):
                key = doc.page_content
                score = weight * (1 / (rank + 60))  # RRF formula
                if key in seen:
                    seen[key][0] += score
                else:
                    seen[key] = [score, doc]
        
        sorted_docs = sorted(seen.values(), key=lambda x: x[0], reverse=True)
        return [doc for _, doc in sorted_docs]

In [28]:
#  Ensemble (Combined)
ensemble_retriever = EnsembleRetriever(retrievers=[vectorstore_retreiver,keyword_retriever],weights=[0.5, 0.5])

In [29]:
# Query
query = "current university"

In [30]:
print("=== Vector Search Results ===")
vector_results = vectorstore_retreiver.invoke(query)
for i, doc in enumerate(vector_results):
    print(f"\n[{i+1}] {doc.page_content[:200]}")

=== Vector Search Results ===

[1] Role: MLOps Engineer
Company: Lyceum Infotech Private Limited
Duration: September 2022 to December 2024
Location: Pune, India

[2] Role: Assistant Professor
Institution: Guru Gobind Singh Foundation
Duration: May 2019 to August 2022
Location: Nashik, India

[3] Secondary Education:
SSC (Secondary School Certificate): 82%
HSC (Higher Secondary Certificate): 57%


COMPLETE TECHNICAL SKILLS


In [31]:
print("\n=== BM25 Keyword Results ===")
keyword_results = keyword_retriever.invoke(query)
for i, doc in enumerate(keyword_results):
    print(f"\n[{i+1}] {doc.page_content[:200]}")


=== BM25 Keyword Results ===

[1] high-impact AI systems that operate at scale and deliver measurable business value.

[2] Rohanta is looking for full-time roles as an ML Engineer, AI Engineer, or NLP Engineer. He aims to work on LLM system architecture, enterprise RAG systems, AI platform engineering, agentic AI workflow

[3] CAREER OBJECTIVE


In [32]:
print("\n=== Ensemble (Combined) Results ===")
ensemble_results = ensemble_retriever.invoke(query)
for i, doc in enumerate(ensemble_results):
    print(f"\n[{i+1}] {doc.page_content[:]}")


=== Ensemble (Combined) Results ===

[1] Role: MLOps Engineer
Company: Lyceum Infotech Private Limited
Duration: September 2022 to December 2024
Location: Pune, India

[2] high-impact AI systems that operate at scale and deliver measurable business value.

[3] Role: Assistant Professor
Institution: Guru Gobind Singh Foundation
Duration: May 2019 to August 2022
Location: Nashik, India

[4] Rohanta is looking for full-time roles as an ML Engineer, AI Engineer, or NLP Engineer. He aims to work on LLM system architecture, enterprise RAG systems, AI platform engineering, agentic AI workflows, evaluation frameworks, and scalable AI infrastructure. His background bridging academia and production engineering makes him a strong candidate for both applied research and senior engineering tracks. He seeks opportunities where he can design high-impact AI systems that operate at scale and

[5] Secondary Education:
SSC (Secondary School Certificate): 82%
HSC (Higher Secondary Certificate): 57%


COM

In [35]:

prompt = ChatPromptTemplate.from_template("""
Answer the question based on the context below.

Context: {context}
Question: {question}
""")

In [36]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [37]:
# Normal chain (vector only)
normal_chain = (
    {"context": vectorstore_retreiver | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [38]:
# Hybrid chain (vector + BM25)
hybrid_chain = (
    {"context": ensemble_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [39]:
# Run
query = "what is current university of rohanta"

print("=== Normal ===")
print(normal_chain.invoke(query))

print("\n=== Hybrid ===")
print(hybrid_chain.invoke(query))

=== Normal ===
The context does not explicitly mention Rohanta's current university. It only mentions that before transitioning into industry, Rohanta taught as an Assistant Professor in Computer Science, but it does not provide information about his current affiliation.

=== Hybrid ===
The current university of Rohanta Bhamare is the Berlin School of Business and Innovation (BSBI), Berlin, Germany.
